# Agente RAG - Challenge ALURA ORACLE OCI
Fluxo:

**5 PDFs → carregamento → chunking → embeddings → vector store → 1 ferramenta de busca → agente → resposta**

## 1. Instalação

In [1]:
%pip install -qU pypdf langchain langchain-community langchain-huggingface langchain-groq langchain-text-splitters sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuração

As chaves ficam em variáveis de ambiente.

In [ ]:
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Defina a variável de ambiente GROQ_API_KEY antes de executar o notebook."
    )

## 3. Modelo de linguagem

O LLM será usado somente para interpretar a pergunta, decidir quando usar a ferramenta e formular a resposta final.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

## 4. Carregando os 5 PDFs

Todos os documentos ficam em `documentos/` e são colocados na **mesma coleção**.
O retriever encontrará os trechos semanticamente mais próximos da pergunta.

In [ ]:
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

DOCUMENTOS_DIR = Path("documentos")

pdfs = sorted(DOCUMENTOS_DIR.glob("*.pdf"))

if len(pdfs) != 5:
    raise ValueError(
        f"Esperados 5 PDFs em '{DOCUMENTOS_DIR}', mas foram encontrados {len(pdfs)}."
    )

for pdf in pdfs:
    print(pdf.name)

loader = DirectoryLoader(
    str(DOCUMENTOS_DIR),
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

pages = loader.load()

print(f"Total de páginas/documentos carregados: {len(pages)}")